# AI-Driven Ocular & Cataract Disease Detection (Medical Cyber-Physical System)
### Complete Google Colab GPU Training & Evaluation Pipeline
**Team Project:** Capstone Phase 1 | **Venue Target:** IEEE Transactions / IEEE Access

This notebook trains the **14-Layer OcuNet-CBAM** architecture on **4,217 Real Clinical Fundus Images** with:
1. **Dual-Stream Preprocessing:** Green-Channel CLAHE + Canny Edge Boundary Detection.
2. **Multi-Class Focal Loss:** Tackling medical class imbalance.
3. **Explainable AI:** Grad-CAM saliency heatmaps for clinical validation.
4. **Full Metrics Suite:** Accuracy, Precision, Recall, Specificity, F1-Score, MCC, and Multi-Class ROC-AUC.

## 1. Environment Setup & GPU Verification

In [ ]:
# Install required packages
!pip install -q kagglehub opencv-python seaborn matplotlib scikit-learn pyyaml

import torch
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] GPU Acceleration Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Active GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("[!] GPU not detected. Go to Runtime -> Change runtime type -> Select T4 GPU for faster training.")

## 2. Automatic Dataset Download & Ingestion (4,217 Real Images)
Dataset in use: **`gunavenkatdoddi/eye-diseases-classification`** on Kaggle
- **Cataract:** 1,038 images
- **Diabetic Retinopathy:** 1,098 images
- **Glaucoma:** 1,007 images
- **Normal Eyes:** 1,074 images

In [ ]:
import os
import shutil
import kagglehub

print("[*] Downloading real-world 4-class Eye Diseases dataset via KaggleHub...")
dataset_cache = kagglehub.dataset_download("gunavenkatdoddi/eye-diseases-classification")
print(f"[+] Dataset downloaded to: {dataset_cache}")

DATA_DIR = "data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

class_mapping = {
    "cataract": "Cataract",
    "diabetic_retinopathy": "Diabetic_Retinopathy",
    "glaucoma": "Glaucoma",
    "normal": "Normal",
}

for c in class_mapping.values():
    os.makedirs(os.path.join(DATA_DIR, c), exist_ok=True)

counts = {c: 0 for c in class_mapping.values()}
for root, dirs, files in os.walk(dataset_cache):
    folder = os.path.basename(root).lower().replace(" ", "_").replace("-", "_")
    for k, v in class_mapping.items():
        if k in folder:
            dest = os.path.join(DATA_DIR, v)
            for f in files:
                if f.lower().endswith((".png", ".jpg", ".jpeg")):
                    src = os.path.join(root, f)
                    dst = os.path.join(dest, f"{v.lower()}_{f}")
                    if not os.path.exists(dst):
                        shutil.copy2(src, dst)
                    counts[v] += 1
            break

print("\n--- Ingested Dataset Summary ---")
for k, v in counts.items():
    print(f"  {k.ljust(22)}: {v} images")
print(f"  Total Images: {sum(counts.values())}")

## 3. Dual-Stream Preprocessing Module

In [ ]:
import cv2
import numpy as np
import torch

class ClinicalEyePreprocessor:
    def __init__(self, target_size=(224, 224), clahe_clip=2.0):
        self.target_size = target_size
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))

    def remove_black_margins(self, img, threshold=10):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
        mask = gray > threshold
        if not np.any(mask): return img
        coords = np.argwhere(mask)
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        return img[y0:y1, x0:x1]

    def process_image(self, img):
        cropped = self.remove_black_margins(img)
        resized = cv2.resize(cropped, self.target_size, interpolation=cv2.INTER_AREA)
        
        # Stream A: Green Channel CLAHE
        green = resized[:, :, 1] if len(resized.shape) == 3 else resized
        ch0 = self.clahe.apply(cv2.GaussianBlur(green, (3, 3), 0.5))
        
        # Stream B: Canny Edge Gradient
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY) if len(resized.shape) == 3 else resized
        ch1 = cv2.Canny(cv2.GaussianBlur(cv2.medianBlur(gray, 3), (3, 3), 0.5), 50, 150)
        
        # Stream C: Enhanced Luminance (L channel)
        if len(resized.shape) == 3:
            lab = cv2.cvtColor(resized, cv2.COLOR_BGR2LAB)
            ch2 = self.clahe.apply(lab[:, :, 0])
        else:
            ch2 = self.clahe.apply(resized)
            
        composite = np.stack([ch0, ch1, ch2], axis=-1).astype(np.float32) / 255.0
        return torch.from_numpy(composite).permute(2, 0, 1).float()

    def preprocess_from_file(self, path):
        img = cv2.imread(path)
        if img is None: raise FileNotFoundError(path)
        return self.process_image(img)

## 4. PyTorch Dataset & DataLoader

In [ ]:
import glob
import random
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

class EyeDataset(Dataset):
    def __init__(self, file_paths, labels, preprocessor, is_training=False):
        self.file_paths = file_paths
        self.labels = labels
        self.preprocessor = preprocessor
        self.is_training = is_training
        self.cache = {}

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        if idx in self.cache:
            tensor = self.cache[idx]
        else:
            try:
                tensor = self.preprocessor.preprocess_from_file(self.file_paths[idx])
            except:
                tensor = torch.zeros((3, 224, 224))
            self.cache[idx] = tensor
            
        if self.is_training:
            tensor = tensor.clone()
            if random.random() < 0.5: tensor = TF.hflip(tensor)
            if random.random() < 0.3: tensor = TF.vflip(tensor)
            if random.random() < 0.7:
                tensor = TF.rotate(tensor, angle=random.choice([15, 30, 45, 90]))
        return tensor, self.labels[idx]

# Stratified Data Splitting
CLASSES = ["Cataract", "Diabetic_Retinopathy", "Glaucoma", "Normal"]
cls2idx = {c: i for i, c in enumerate(CLASSES)}

all_files, all_labels = [], []
for c in CLASSES:
    files = glob.glob(f"{DATA_DIR}/{c}/*.*")
    all_files.extend(files)
    all_labels.extend([cls2idx[c]] * len(files))

random.seed(42)
train_f, train_l, val_f, val_l, test_f, test_l = [], [], [], [], [], []
for c_idx in range(4):
    idxes = [i for i, l in enumerate(all_labels) if l == c_idx]
    random.shuffle(idxes)
    n = len(idxes)
    nt, nv = int(n * 0.70), int(n * 0.15)
    for i in idxes[:nt]: train_f.append(all_files[i]); train_l.append(all_labels[i])
    for i in idxes[nt:nt+nv]: val_f.append(all_files[i]); val_l.append(all_labels[i])
    for i in idxes[nt+nv:]: test_f.append(all_files[i]); test_l.append(all_labels[i])

prep = ClinicalEyePreprocessor()
train_loader = DataLoader(EyeDataset(train_f, train_l, prep, True), batch_size=32, shuffle=True)
val_loader = DataLoader(EyeDataset(val_f, val_l, prep, False), batch_size=32, shuffle=False)
test_loader = DataLoader(EyeDataset(test_f, test_l, prep, False), batch_size=32, shuffle=False)
print(f"[*] Train: {len(train_f)} | Val: {len(val_f)} | Test: {len(test_f)}")

## 5. Novel 14-Layer OcuNet-CBAM Architecture & Multi-Class Focal Loss

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    def __init__(self, in_c, r=16):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_c, in_c // r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_c // r, in_c, bias=False)
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        avg_pool = torch.mean(x, dim=(2, 3))
        max_pool = torch.amax(x, dim=(2, 3))
        w = torch.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool)).view(b, c, 1, 1)
        return x * w

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        avg_p = torch.mean(x, dim=1, keepdim=True)
        max_p, _ = torch.max(x, dim=1, keepdim=True)
        w = torch.sigmoid(self.conv(torch.cat([avg_p, max_p], dim=1)))
        return x * w

class CBAM(nn.Module):
    def __init__(self, in_c):
        super().__init__()
        self.ca = ChannelAttention(in_c)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

class OcuNetCBAM(nn.Module):
    def __init__(self, num_classes=4, f=32):
        super().__init__()
        self.stage1 = nn.Sequential(
            nn.Conv2d(3, f, 3, padding=1, bias=False), nn.BatchNorm2d(f), nn.ReLU(),
            nn.Conv2d(f, f, 3, padding=1, bias=False), nn.BatchNorm2d(f), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.stage2 = nn.Sequential(
            nn.Conv2d(f, f*2, 3, padding=1, bias=False), nn.BatchNorm2d(f*2), nn.ReLU(),
            nn.Conv2d(f*2, f*2, 3, padding=1, bias=False), nn.BatchNorm2d(f*2), nn.ReLU(),
            CBAM(f*2), nn.MaxPool2d(2, 2)
        )
        self.stage3 = nn.Sequential(
            nn.Conv2d(f*2, f*4, 3, padding=1, bias=False), nn.BatchNorm2d(f*4), nn.ReLU(),
            nn.Conv2d(f*4, f*4, 3, padding=1, bias=False), nn.BatchNorm2d(f*4), nn.ReLU(),
            CBAM(f*4), nn.MaxPool2d(2, 2)
        )
        self.stage4 = nn.Sequential(
            nn.Conv2d(f*4, f*8, 3, padding=1, bias=False), nn.BatchNorm2d(f*8), nn.ReLU(),
            nn.Conv2d(f*8, f*8, 3, padding=1, bias=False), nn.BatchNorm2d(f*8), nn.ReLU(),
            CBAM(f*8)
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(f*8, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        return self.head(x)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = OcuNetCBAM(num_classes=4).to(device)
params = sum(p.numel() for p in model.parameters())
print(f"[*] Total Parameters: {params:,} (Size: {params*4/(1024*1024):.2f} MB)")

## 6. Model Training & Evaluation Engine (GPU Accelerated)

In [ ]:
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

EPOCHS = 15
criterion = FocalLoss(gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f"[*] Starting GPU Training for {EPOCHS} epochs...")
start_t = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, t_corr, t_tot = 0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, lbls)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        t_corr += (preds.argmax(1) == lbls).sum().item()
        t_tot += lbls.size(0)
    scheduler.step()
    
    # Validation
    model.eval()
    v_loss, v_corr, v_tot = 0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = model(imgs)
            loss = criterion(preds, lbls)
            v_loss += loss.item() * imgs.size(0)
            v_corr += (preds.argmax(1) == lbls).sum().item()
            v_tot += lbls.size(0)
            
    tr_acc, va_acc = t_corr / t_tot, v_corr / v_tot
    history['train_loss'].append(t_loss / t_tot)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(v_loss / v_tot)
    history['val_acc'].append(va_acc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train Acc: {tr_acc*100:.2f}% | Val Acc: {va_acc*100:.2f}%")
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(model.state_dict(), 'ocunet_cbam_best.pth')

print(f"\n[+] Training complete in {time.time()-start_t:.2f}s. Best Val Accuracy: {best_acc*100:.2f}%")

## 7. Test Set Evaluation & Diagnostic Metrics

In [ ]:
model.load_state_dict(torch.load('ocunet_cbam_best.pth'))
model.eval()

y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)
        y_true.extend(lbls.numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro')
rec = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')
mcc = matthews_corrcoef(y_true, y_pred)
auc_roc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')

print("\n=======================================================")
print("          CLINICAL TEST EVALUATION SUMMARY              ")
print("=======================================================")
print(f"  Accuracy            : {acc*100:.2f}%")
print(f"  Precision (Macro)   : {prec*100:.2f}%")
print(f"  Recall / Sensitivity: {rec*100:.2f}%")
print(f"  F1-Score (Macro)    : {f1*100:.2f}%")
print(f"  Matthews Corr (MCC) : {mcc:.4f}")
print(f"  Multi-Class ROC-AUC : {auc_roc:.4f}")
print("=======================================================")

# Confusion Matrix Plot
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Test Confusion Matrix (OcuNet-CBAM)')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.tight_layout()
plt.show()

## 8. Explainable AI (Grad-CAM Saliency Maps)

In [ ]:
# Grad-CAM implementation
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        self.target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))
        
    def generate(self, x, target_cls):
        self.model.eval()
        self.model.zero_grad()
        out = self.model(x)
        out[0, target_cls].backward()
        alpha = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = F.relu(torch.sum(alpha * self.activations, dim=1, keepdim=True)).squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

grad_cam = GradCAM(model, model.stage4[-1])

plt.figure(figsize=(14, 4))
for i, cls_name in enumerate(CLASSES):
    img_path = glob.glob(f"{DATA_DIR}/{cls_name}/*.*")[0]
    raw = cv2.imread(img_path)
    tensor = prep.process_image(raw).unsqueeze(0).to(device)
    tensor.requires_grad = True
    cam = grad_cam.generate(tensor, i)
    
    cam_resized = cv2.resize(cam, (raw.shape[1], raw.shape[0]))
    overlay = cv2.addWeighted(raw, 0.5, cv2.applyColorMap(np.uint8(255*cam_resized), cv2.COLORMAP_JET), 0.5, 0)
    
    plt.subplot(1, 4, i+1)
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title(f"{cls_name}\nGrad-CAM Focus")
    plt.axis('off')

plt.tight_layout()
plt.show()